In [1]:
import pandas as pd
file = "yellow_tripdata_2026-01.parquet"
df = pd.read_parquet(file)
print(df.head(5))

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2026-01-01 00:54:04   2026-01-01 00:59:37              1.0   
1         1  2026-01-01 00:34:04   2026-01-01 00:39:47              0.0   
2         1  2026-01-01 00:57:06   2026-01-01 01:05:59              0.0   
3         2  2026-01-01 00:15:22   2026-01-01 00:58:10              4.0   
4         2  2026-01-01 00:27:13   2026-01-01 00:40:43              0.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           0.97         1.0                  N           239           238   
1           0.90         1.0                  N           163           162   
2           1.40         1.0                  N            43           237   
3           5.58         1.0                  N           142           209   
4           2.16         1.0                  N            88           144   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


# Pyspark vs Pandas

In [2]:

import time
import psutil
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg

file = "yellow_tripdata_2026-01.parquet"

process = psutil.Process(os.getpid())

# ======================
# 🐼 PANDAS
# ======================
mem_before = process.memory_info().rss / 1024 / 1024
start = time.time()

df_pandas = pd.read_parquet(file)
avg_fare_pandas = df_pandas["fare_amount"].mean()

end = time.time()
mem_after = process.memory_info().rss / 1024 / 1024

print("PANDAS")
print("Avg fare:", avg_fare_pandas)
print("Temps:", round(end - start, 2), "s")
print("RAM:", round(mem_after - mem_before, 2), "MB")

print("##########################################################")

# ======================
# ⚡ PYSPARK
# ======================

# démarrage Spark (hors mesure)
spark = SparkSession.builder \
    .appName("compare") \
    .master("local[*]") \
    .getOrCreate()

start = time.time()

df_spark = spark.read.parquet(file)
avg_fare_spark = df_spark.select(avg("fare_amount")).collect()[0][0]

end = time.time()

print("PYSPARK")
print("Avg fare:", avg_fare_spark)
print("Temps:", round(end - start, 2), "s")
print("RAM: non fiable (Spark utilise JVM)")

spark.stop()

PANDAS
Avg fare: 20.804253893203256
Temps: 0.29 s
RAM: 602.01 MB
##########################################################


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 10:07:37 WARN Utils: Your hostname, PYTHON-08, resolves to a loopback address: 127.0.1.1; using 172.22.114.75 instead (on interface eno1)
26/04/17 10:07:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 10:07:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PYSPARK
Avg fare: 20.804253893199448
Temps: 3.33 s
RAM: non fiable (Spark utilise JVM)


# RDD et lazy evaluation 

In [3]:
from pyspark import SparkContext


#sc = SparkContext("local", "Lazy Test")
sc = SparkContext("local", "RDD example")

l = [1, 2, 3, 4, 5]

rdd = sc.parallelize(l)

rdd2 = rdd.map(lambda x: x * 2)

#print(rdd)
print(rdd.collect())
time.sleep(20)
sc.stop()



[1, 2, 3, 4, 5]


### 12. Explorer glom() (distribution par partition)

In [4]:

sc = SparkContext("local[2]", "glom_demo")

rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8], 3)
print(rdd.glom().collect())
rdd2 = rdd.map(lambda x: x * 10)

print(rdd2.glom().collect())
time.sleep(20)
sc.stop()

[[1, 2], [3, 4], [5, 6, 7, 8]]
[[10, 20], [30, 40], [50, 60, 70, 80]]


### 13. Passer à [1,2,3,4,5]*1_000_000 – tester map, filter, reduce

In [5]:

sc = SparkContext("local[*]", "big_rdd_test")

data = [1, 2, 3, 4, 5] * 1_000_000

rdd = sc.parallelize(data)
rdd_map = rdd.map(lambda x: x * 2)
rdd_filter = rdd_map.filter(lambda x: x > 5)
result = rdd_filter.reduce(lambda a, b: a + b)

print(result)

26/04/17 10:10:06 WARN TaskSetManager: Stage 0 contains a task of very large size (1246 KiB). The maximum recommended task size is 1000 KiB.


24000000


In [7]:
sc.stop()

### Challenge – Arbres remarquables de Paris
### 14. Télécharger le dataset (lien ci-dessous) et répondre en utilisant uniquement l'API RDD :
### ◦ Quel est l'âge du plus vieil arbre ?
### ◦ Quelle est la hauteur moyenne de tous les arbres ?
### ◦ Dans quel arrondissement y a-t-il le plus d'arbres ?

In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ArbresRemarquables").getOrCreate()
sc = spark.sparkContext

# Charger le parquet via DataFrame puis convertir en RDD
df = spark.read.parquet("arbresremarquablesparis.parquet")
rdd = df.rdd

# Afficher les colonnes disponibles
print(df.columns)

['geom_x_y', 'arbres_idbase', 'arbres_domanialite', 'arbres_arrondissement', 'arbres_complementadresse', 'arbres_numero', 'arbres_adresse', 'arbres_circonferenceencm', 'arbres_hauteurenm', 'arbres_stadedeveloppement', 'arbres_pepiniere', 'arbres_genre', 'arbres_espece', 'arbres_varieteoucultivar', 'arbres_dateplantation', 'arbres_libellefrancais', 'com_idbase', 'com_idarbre', 'com_site', 'com_adresse', 'com_complement_adresse', 'com_arrondissement', 'com_domanialite', 'com_nom_usuel', 'com_nom_latin', 'com_autorite_taxo', 'com_annee_plantation', 'com_qualification_rem', 'com_resume', 'com_descriptif', 'com_delib_num', 'com_delib_date', 'com_label_arbres', 'com_url_pdf', 'com_url_photo1', 'com_copyright1']


In [11]:
sc.stop()

In [14]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.appName("ArbresRemarquables").getOrCreate()
sc = spark.sparkContext
df = spark.read.parquet("arbresremarquablesparis.parquet")
rdd = df.rdd

annee_actuelle = 2026

# ── 1. Âge du plus vieil arbre ──────────────────────────────────────────────
def annee_valide(row):
    val = row["com_annee_plantation"]
    if val is None:
        return False
    try:
        int(val)
        return True
    except (ValueError, TypeError):
        return False

age_max = (
    rdd
    .filter(annee_valide)
    .map(lambda row: annee_actuelle - int(row["com_annee_plantation"]))
    .reduce(lambda a, b: a if a > b else b)
)
print(f"Âge du plus vieil arbre : {age_max} ans")


# ── 2. Hauteur moyenne de tous les arbres ───────────────────────────────────
def hauteur_valide(row):
    val = row["arbres_hauteurenm"]
    if val is None:
        return False
    try:
        float(val)
        return True
    except (ValueError, TypeError):
        return False

rdd_hauteur = (
    rdd
    .filter(hauteur_valide)
    .map(lambda row: (float(row["arbres_hauteurenm"]), 1))
)

total_hauteur, total_count = rdd_hauteur.reduce(lambda a, b: (a[0]+b[0], a[1]+b[1]))
moyenne = total_hauteur / total_count
print(f"Hauteur moyenne : {moyenne:.2f} m")


# ── 3. Arrondissement avec le plus d'arbres ─────────────────────────────────
def arrond_valide(row):
    val = row["arbres_arrondissement"]
    if val is None:
        return False
    try:
        str(val).strip()
        return True
    except (ValueError, TypeError):
        return False

arrond_max = (
    rdd
    .filter(arrond_valide)
    .map(lambda row: (row["arbres_arrondissement"], 1))
    .reduceByKey(lambda a, b: a + b)
    .reduce(lambda a, b: a if a[1] > b[1] else b)
)
print(f"Arrondissement avec le plus d'arbres : {arrond_max[0]} ({arrond_max[1]} arbres)")

Âge du plus vieil arbre : 424 ans
Hauteur moyenne : 20.35 m
Arrondissement avec le plus d'arbres : BOIS DE VINCENNES (32 arbres)


In [6]:
sc.stop()

###  5 – DataFrames Spark – nettoyage et analyses 

###  Chargement et nettoyage
###  15. Charger le PARQUET taxi : spark.read.parquet("taxi.parquet", inferSchema=True, header=True)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

# Recréer la session
spark = SparkSession.builder.appName("Taxi").getOrCreate()
sc = spark.sparkContext
df = spark.read.parquet("yellow_tripdata_2026-01.parquet")
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 12:12:49 WARN Utils: Your hostname, PYTHON-08, resolves to a loopback address: 127.0.1.1; using 172.22.114.75 instead (on interface eno1)
26/04/17 12:12:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 12:12:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/17 12:12:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/17 12:12:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

###  16. Afficher le schéma et corriger les types si nécessaire avec un StructType manuel


In [2]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



###  17. Compter les valeurs nulles par colonne et décider d'une stratégie (dropna ou fillna)


In [15]:
from pyspark.sql.functions import col, sum as spark_sum

df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
]).show(vertical=True)

# On vérifie combien de lignes total
print(f"Total lignes : {df.count()}")
# Si 1 088 058 représente une grosse partie → fillna
# Si c'est petit → dropna

-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 1088058 
 trip_distance         | 0       
 RatecodeID            | 1088058 
 store_and_fwd_flag    | 1088058 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 1088058 
 Airport_fee           | 1088058 
 cbd_congestion_fee    | 0       

Total lignes : 3724889


In [16]:
df.groupBy("VendorID").count().show()

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 710425|
|       7|  44705|
|       2|2965742|
|       6|   4017|
+--------+-------+



In [17]:
from pyspark.sql.functions import col

df.filter(col("passenger_count").isNull()) \
  .groupBy("VendorID") \
  .count() \
  .show()

+--------+------+
|VendorID| count|
+--------+------+
|       1|139406|
|       6|  4017|
|       2|944635|
+--------+------+



### Observations :

### Vendor 6 → 100% de ses lignes ont des nulls, il ne remplit jamais ces champs
### Vendor 2 → presque 1 trajet sur 3 sans ces champs
### Vendor 1 → 1 trajet sur 5 sans ces champs
### Vendor 7 → 0 null, il remplit toujours tout 



In [31]:
# Stratégie fillna (recommandée ici car beaucoup de nulls)
df_clean = df.fillna({
    "passenger_count": 1,          # 1 passager par défaut
    "RatecodeID": 1,               # tarif standard par défaut
    "store_and_fwd_flag": "N",     # N = non stocké par défaut
    "congestion_surcharge": 0.0,   # pas de surcharge par défaut
    "Airport_fee": 0.0             # pas de frais aéroport par défaut
})

print(f"Nulls restants après fillna : {df_clean.filter(col('passenger_count').isNull()).count()}")

Nulls restants après fillna : 0


###  18. Filtrer les valeurs aberrantes : fare_amount ≤ 0 ou > 500$, trip_distance = 0 ou > 200 miles


In [32]:
print(f"Avant filtre : {df_clean.count()} lignes")

df_clean = df_clean.filter(
    (col("fare_amount") > 0) & (col("fare_amount") <= 500) &
    (col("trip_distance") > 0) & (col("trip_distance") <= 200)
)

print(f"Après filtre aberrants : {df_clean.count()} lignes")

Avant filtre : 3724889 lignes
Après filtre aberrants : 3560706 lignes


###  19. Dédupliquer : df.dropDuplicates()

In [27]:
df_clean = df_clean.dropDuplicates()
print(f"Après déduplification : {df_clean.count()} lignes")

Après déduplification : 3560706 lignes


###  20 – Pourcentage conservé

In [21]:

total = df.count()
conserve = df_clean.count()
print(f"Total initial     : {total} lignes")
print(f"Total conservé    : {conserve} lignes")
print(f"Lignes conservées : {conserve / total * 100:.2f}%")

Total initial     : 3724889 lignes
Total conservé    : 3560706 lignes
Lignes conservées : 95.59%


## Challenges d'analyse – API DataFrame

### 21. Quelle est la course la plus chère (fare_amount) ? La plus longue (trip_distance) ?

In [22]:
# La plus chère
df_clean.orderBy(col("fare_amount").desc()).select(
    "fare_amount", "trip_distance", "tpep_pickup_datetime"
).show(1)

# La plus longue
df_clean.orderBy(col("trip_distance").desc()).select(
    "fare_amount", "trip_distance", "tpep_pickup_datetime"
).show(1)

+-----------+-------------+--------------------+
|fare_amount|trip_distance|tpep_pickup_datetime|
+-----------+-------------+--------------------+
|      500.0|        98.64| 2026-01-08 17:59:42|
+-----------+-------------+--------------------+
only showing top 1 row


+-----------+-------------+--------------------+
|fare_amount|trip_distance|tpep_pickup_datetime|
+-----------+-------------+--------------------+
|       33.5|        192.8| 2026-01-08 11:25:05|
+-----------+-------------+--------------------+
only showing top 1 row


### 22. Quel est le montant moyen par type de paiement ? Classer par montant décroissant.

In [23]:
from pyspark.sql.functions import avg, round as spark_round

df_clean.groupBy("payment_type") \
    .agg(spark_round(avg("total_amount"), 2).alias("montant_moyen")) \
    .orderBy(col("montant_moyen").desc()) \
    .show()

+------------+-------------+
|payment_type|montant_moyen|
+------------+-------------+
|           0|        31.51|
|           4|        29.75|
|           1|         29.4|
|           3|        25.06|
|           2|        25.06|
+------------+-------------+



###  23. Quelles sont les 10 zones de prise en charge les plus actives ?

In [24]:
df_clean.groupBy("PULocationID") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)

+------------+------+
|PULocationID| count|
+------------+------+
|         237|155782|
|         236|149123|
|         132|142787|
|         161|141586|
|         186|107392|
|         142|106731|
|         162|105344|
|         230|102092|
|          79| 97104|
|         239| 94153|
+------------+------+
only showing top 10 rows


### 24. Quelle est la distribution des courses par heure de la journée ?

In [25]:
from pyspark.sql.functions import hour

df_clean.withColumn("heure", hour("tpep_pickup_datetime")) \
    .groupBy("heure") \
    .count() \
    .orderBy("heure") \
    .show(24)

+-----+------+
|heure| count|
+-----+------+
|    0|109334|
|    1| 75896|
|    2| 52621|
|    3| 37544|
|    4| 27727|
|    5| 31915|
|    6| 62343|
|    7|111522|
|    8|145738|
|    9|153931|
|   10|158159|
|   11|168125|
|   12|183346|
|   13|191312|
|   14|202754|
|   15|216486|
|   16|206343|
|   17|224131|
|   18|232812|
|   19|211578|
|   20|208570|
|   21|208340|
|   22|190933|
|   23|149246|
+-----+------+



### 25. Enregistrer en vue temporaire (createOrReplaceTempView) et reproduire une requête via SparkSQL

In [26]:
# Créer la vue
df_clean.createOrReplaceTempView("taxi")

# Reproduire la Q22 en SQL pur
spark.sql("""
    SELECT payment_type,
           ROUND(AVG(total_amount), 2) AS montant_moyen
    FROM taxi
    GROUP BY payment_type
    ORDER BY montant_moyen DESC
""").show()

+------------+-------------+
|payment_type|montant_moyen|
+------------+-------------+
|           0|        31.51|
|           4|        29.75|
|           1|         29.4|
|           3|        25.06|
|           2|        25.06|
+------------+-------------+

